# Emergency Green Corridor System - Full Stack

This notebook contains the complete code for the Emergency Green Corridor System.
Run the cells sequentially to:
1. Install necessary libraries.
2. Initialize the backend logic (City Graph, Traffic Model, Routes).
3. Generate the required frontend files (HTML, CSS, JS).
4. Start the Flask Server.


In [1]:
%pip install flask networkx pandas scikit-learn fuzzywuzzy python-Levenshtein

In [2]:
import os
import json
import random
import time
import networkx as nx
import pandas as pd
import numpy as np
from flask import Flask, render_template, request, jsonify, redirect, url_for, session
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from fuzzywuzzy import process

app = Flask(__name__)
app.secret_key = 'green_coridor_secret_key'  # Change this in production

# --- DATA & MODEL SETUP (From Notebook) ---

def create_city_graph():
    G = nx.Graph()
    # EXPANDED COIMBATORE NODES (Approx Coordinate Mapping for Visualization)
    nodes = {
        "Airport": (12, 12), "Sitra": (11, 11), "Hopes College": (9, 10), "Peelamedu": (8, 9),
        "Lakshmi Mills": (6, 6), "Gandhipuram": (4, 5), "Ukkadam": (2, 2), "Railway Station": (3, 4),
        "Government Hospital": (3, 3), "RS Puram": (2, 6), "Saibaba Colony": (3, 8), "Ganapathy": (5, 8),
        "Saravanampatti": (8, 12), "Singanallur": (8, 4), "Ramanathapuram": (6, 3), "Podanur": (4, 1),
        "Marudhamalai": (0, 10), "Vadavalli": (1, 8), "Thudiyalur": (4, 10), "Koundampalayam": (3, 9),
        "Town Hall": (2, 3.5), "Race Course": (3.5, 4.5), "Brookefields": (2.5, 5), "Cross Cut Road": (4.2, 5.2),
        "100 Feet Road": (4.5, 5.5), "Fun Mall": (8.2, 8.8), "Prozone Mall": (6, 9), "Tidel Park": (8.5, 9.5),
        "Codissia": (10, 11), "Sulur": (13, 8), "Sundarapuram": (3, 0.5)
    }
    for name, pos in nodes.items():
        G.add_node(name, pos=pos)

    # ROADS (Edges with approx travel time in mins)
    G.add_edge("Airport", "Sitra", weight=3)
    G.add_edge("Sitra", "Codissia", weight=2)
    G.add_edge("Sitra", "Hopes College", weight=5)
    G.add_edge("Hopes College", "Tidel Park", weight=2)
    G.add_edge("Hopes College", "Peelamedu", weight=4)
    G.add_edge("Peelamedu", "Fun Mall", weight=2)
    G.add_edge("Peelamedu", "Lakshmi Mills", weight=6)
    G.add_edge("Lakshmi Mills", "Gandhipuram", weight=7)
    G.add_edge("Lakshmi Mills", "Ramanathapuram", weight=5)
    G.add_edge("Lakshmi Mills", "Race Course", weight=4)
    G.add_edge("Gandhipuram", "Cross Cut Road", weight=2)
    G.add_edge("Gandhipuram", "100 Feet Road", weight=3)
    G.add_edge("Gandhipuram", "Ganapathy", weight=6)
    G.add_edge("Ganapathy", "Prozone Mall", weight=4)
    G.add_edge("Ganapathy", "Saravanampatti", weight=8)
    G.add_edge("Saravanampatti", "Thudiyalur", weight=10)
    G.add_edge("Thudiyalur", "Koundampalayam", weight=5)
    G.add_edge("Koundampalayam", "Saibaba Colony", weight=4)
    G.add_edge("Gandhipuram", "RS Puram", weight=7)
    G.add_edge("RS Puram", "Brookefields", weight=3)
    G.add_edge("RS Puram", "Saibaba Colony", weight=4)
    G.add_edge("Saibaba Colony", "Vadavalli", weight=6)
    G.add_edge("Vadavalli", "Marudhamalai", weight=10)
    G.add_edge("Gandhipuram", "Railway Station", weight=5)
    G.add_edge("Railway Station", "Government Hospital", weight=2)
    G.add_edge("Railway Station", "Town Hall", weight=3)
    G.add_edge("Government Hospital", "Ukkadam", weight=5)
    G.add_edge("Town Hall", "Ukkadam", weight=4)
    G.add_edge("Ukkadam", "Ramanathapuram", weight=6)
    G.add_edge("Ramanathapuram", "Singanallur", weight=5)
    G.add_edge("Singanallur", "Sulur", weight=12)
    G.add_edge("Singanallur", "Hopes College", weight=7)
    G.add_edge("Ramanathapuram", "Podanur", weight=8)
    G.add_edge("Podanur", "Sundarapuram", weight=6)
    G.add_edge("Ukkadam", "Sundarapuram", weight=7)
    return G

# Global Graph
city_graph = create_city_graph()
node_list = list(city_graph.nodes())

# Train Model Global
rf_model = None

def train_model():
    global rf_model
    def generate_training_data(n_samples=5000):
        data = []
        nodes = list(city_graph.nodes())
        for _ in range(n_samples):
            hour = random.randint(0, 23)
            is_weekend = random.choice([0, 1])
            loc_id = random.randint(0, len(nodes)-1)
            weather = random.randint(0, 2)

            congestion = 0
            if (8 <= hour <= 10) or (17 <= hour <= 21): congestion += 1
            if weather > 0: congestion += 1
            if nodes[loc_id] in ["Gandhipuram", "Lakshmi Mills", "Ukkadam", "100 Feet Road", "Cross Cut Road"]:
                congestion += 1
            congestion = min(2, congestion)
            if is_weekend: congestion = max(0, congestion - 1)
            if random.random() < 0.12:
                congestion = random.randint(0, 2)
            data.append([hour, is_weekend, loc_id, weather, congestion])
        return pd.DataFrame(data, columns=['Hour', 'Is_Weekend', 'LocationID', 'Weather', 'Target'])

    if rf_model is None:
        print("Training Traffic Prediction Model...")
        df_train = generate_training_data(8000)
        X = df_train[['Hour', 'Is_Weekend', 'LocationID', 'Weather']]
        y = df_train['Target']

        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
        rf_model.fit(X_train, y_train)

        from sklearn.metrics import accuracy_score
        acc = accuracy_score(y_test, rf_model.predict(X_test))
        print(f"Model Trained. Accuracy: {acc:.4f}")
        return acc
    return 0.0

model_accuracy = train_model()

@app.route('/api/accuracy')
def get_accuracy():
    return jsonify({'accuracy': f"{model_accuracy*100:.1f}%"})

def predict_congestion(location_name):
    nodes = list(city_graph.nodes())
    try:
        loc_id = nodes.index(location_name)
    except ValueError:
        loc_id = 0
    # Simulate current time: 18:00, weekday, clear weather
    pred = rf_model.predict([[18, 0, loc_id, 0]])[0]
    return ["LOW", "MEDIUM", "HIGH"][pred]

# Users DB (In-memory for simplicity, or json)
USERS_FILE = 'users.json'

def load_users():
    if os.path.exists(USERS_FILE):
        with open(USERS_FILE, 'r') as f:
            return json.load(f)
    return {}

def save_users(users):
    with open(USERS_FILE, 'w') as f:
        json.dump(users, f)

# --- ROUTES ---

@app.route('/')
def home():
    if 'user' in session:
        return redirect(url_for('dashboard'))
    return redirect(url_for('login_page'))

@app.route('/login', methods=['GET', 'POST'])
def login_page():
    if request.method == 'POST':
        data = request.json
        username = data.get('username')
        password = data.get('password')
        users = load_users()
        if username in users and users[username]['password'] == password:
            session['user'] = username
            return jsonify({'success': True})
        return jsonify({'success': False, 'message': 'Invalid credentials'})
    return render_template('login.html')

@app.route('/register', methods=['POST'])
def register():
    data = request.json
    username = data.get('username')
    password = data.get('password')
    users = load_users()
    if username in users:
        return jsonify({'success': False, 'message': 'User already exists'})
    users[username] = {'password': password}
    save_users(users)
    return jsonify({'success': True})

@app.route('/dashboard')
def dashboard():
    if 'user' not in session:
        return redirect(url_for('login_page'))
    return render_template('dashboard.html', user=session['user'])

@app.route('/logout')
def logout():
    session.pop('user', None)
    return redirect(url_for('login_page'))

@app.route('/api/locations')
def get_locations():
    # Return locations and their positions for the map
    nodes_data = []
    positions = nx.get_node_attributes(city_graph, 'pos')
    for node in city_graph.nodes():
        nodes_data.append({
            "name": node,
            "coords": positions[node] # Tuple (x, y) - abstract coords
        })
    return jsonify(nodes_data)

@app.route('/api/route', methods=['POST'])
def get_route():
    data = request.json
    start_input = data.get('start')
    end_input = data.get('end')

    nodes = list(city_graph.nodes())
    start_match = process.extractOne(start_input, nodes)
    end_match = process.extractOne(end_input, nodes)

    if not start_match or start_match[1] < 60:
         return jsonify({'success': False, 'message': f"Start location '{start_input}' not found."})
    if not end_match or end_match[1] < 60:
         return jsonify({'success': False, 'message': f"End location '{end_input}' not found."})

    start_node = start_match[0]
    end_node = end_match[0]

    try:
        path = nx.shortest_path(city_graph, start_node, end_node, weight="weight")
        original_time = nx.shortest_path_length(city_graph, start_node, end_node, weight="weight")
    except nx.NetworkXNoPath:
        return jsonify({'success': False, 'message': "No route found."})

    # Traffic Logic
    total_saved = 0
    signals_cleared = 0
    route_details = []

    for i in range(len(path)-1):
        u, v = path[i], path[i+1]
        traffic = predict_congestion(v) # Predict traffic at the next node/signal

        # Get edge weight
        orig_weight = city_graph[u][v]['weight']

        reduction = 0.6 if traffic == "HIGH" else 0.3
        saved = orig_weight * reduction
        total_saved += saved
        signals_cleared += 1

        route_details.append({
            "node": v,
            "traffic": traffic,
            "saved": round(saved, 1)
        })

    new_time = max(1, original_time - total_saved)

    return jsonify({
        'success': True,
        'start_corrected': start_node,
        'end_corrected': end_node,
        'path': path,
        'original_time': original_time,
        'new_time': round(new_time, 1),
        'saved_time': round(total_saved, 1),
        'signals_cleared': signals_cleared,
        'details': route_details
    })



Training Traffic Prediction Model...
Model Trained. Accuracy: 0.9187


In [3]:
import os
os.makedirs('templates', exist_ok=True)
file_path = os.path.join('templates', 'login.html')
content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Green Corridor - Login</title>
    <link rel="preconnect" href="https://fonts.googleapis.com">
    <link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>
    <link href="https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;600;700&display=swap" rel="stylesheet">
    <link rel="stylesheet" href="{{ url_for('static', filename='css/style.css') }}">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0/css/all.min.css">
</head>
<body class="login-body">
    <div class="background-globes">
        <div class="globe globe-1"></div>
        <div class="globe globe-2"></div>
        <div class="globe globe-3"></div>
    </div>

    <div class="login-container glass-panel">
        <div class="login-header">
            <h1><i class="fa-solid fa-traffic-light"></i> Green Corridor</h1>
            <p>AI-Powered Emergency Response Optimization</p>
        </div>

        <div class="form-toggle">
            <button id="loginBtn" class="toggle-btn active" onclick="showForm('login')">Login</button>
            <button id="registerBtn" class="toggle-btn" onclick="showForm('register')">Register</button>
        </div>

        <!-- Login Form -->
        <form id="loginForm" class="auth-form">
            <div class="input-group">
                <i class="fa-solid fa-user"></i>
                <input type="text" id="loginUsername" placeholder="Username" required>
            </div>
            <div class="input-group">
                <i class="fa-solid fa-lock"></i>
                <input type="password" id="loginPassword" placeholder="Password" required>
            </div>
            <button type="submit" class="cta-button">Access System <i class="fa-solid fa-arrow-right"></i></button>
        </form>

        <!-- Register Form -->
        <form id="registerForm" class="auth-form hidden">
            <div class="input-group">
                <i class="fa-solid fa-user-plus"></i>
                <input type="text" id="regUsername" placeholder="Choose Username" required>
            </div>
            <div class="input-group">
                <i class="fa-solid fa-lock"></i>
                <input type="password" id="regPassword" placeholder="Choose Password" required>
            </div>
            <button type="submit" class="cta-button secondary">Create Account</button>
        </form>

        <div id="message" class="message"></div>
    </div>

    <script src="{{ url_for('static', filename='js/script.js') }}"></script>
</body>
</html>
"""
with open(file_path, 'w', encoding='utf-8') as f:
    f.write(content)
print(f'Successfully generated {file_path}')

Successfully generated templates/login.html


In [4]:
import os
os.makedirs('templates', exist_ok=True)
file_path = os.path.join('templates', 'dashboard.html')
content = """<!DOCTYPE html>
<html lang="en">

<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Green Corridor - Dashboard</title>
    <link href="https://fonts.googleapis.com/css2?family=Outfit:wght@300;400;600;700&display=swap" rel="stylesheet">
    <link rel="stylesheet" href="{{ url_for('static', filename='css/style.css') }}">
    <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0/css/all.min.css">

    <!-- Leaflet CSS -->
    <link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css"
        integrity="sha256-p4NxAoJBhIIN+hmNHrzRCf9tD/miZyoHS5obTRR9BMY=" crossorigin="" />
</head>

<body class="dashboard-body">
    <nav class="navbar glass-panel">
        <div class="nav-brand">
            <i class="fa-solid fa-traffic-light"></i> <span>Green Corridor System</span>
        </div>
        <div class="nav-user">
            <span>Welcome, {{ user }}</span>
            <a href="/logout" class="logout-btn"><i class="fa-solid fa-sign-out-alt"></i></a>
        </div>
    </nav>

    <div class="dashboard-container">
        <!-- Sidebar Controls -->
        <aside class="sidebar glass-panel">
            <h2><i class="fa-solid fa-route"></i> Route Planner</h2>
            <div class="control-group">
                <label>Start Location</label>
                <div class="input-wrapper">
                    <i class="fa-solid fa-map-marker-alt"></i>
                    <input type="text" id="startLocation" placeholder="e.g. Hopes College" list="locationsList">
                </div>
            </div>

            <div class="control-group">
                <label>Destination</label>
                <div class="input-wrapper">
                    <i class="fa-solid fa-flag-checkered"></i>
                    <input type="text" id="endLocation" placeholder="e.g. Gandhipuram" list="locationsList">
                </div>
            </div>

            <datalist id="locationsList">
                <!-- Populated by JS -->
            </datalist>

            <button id="activateSystemBtn" class="activate-btn">
                <i class="fa-solid fa-bolt"></i> ACTIVATE GREEN CORRIDOR
            </button>

            <div id="routeStats" class="stats-panel hidden">
                <h3>Route Analysis</h3>
                <div class="stat-item">
                    <span>Signals Cleared</span>
                    <strong id="signalsCount">0</strong>
                </div>
                <div class="stat-item">
                    <span>Model Accuracy</span>
                    <strong id="modelAcc" class="highlight">--%</strong>
                </div>
                <div class="stat-item">
                    <span>Time Saved</span>
                    <strong id="timeSaved" class="highlight">0 mins</strong>
                </div>
                <div class="stat-item">
                    <span>Est. Duration</span>
                    <strong id="newTime">0 mins</strong>
                </div>
            </div>

            <div id="routeLogs" class="logs-panel">
                <!-- Logs will appear here -->
            </div>
        </aside>

        <!-- Main Map Area -->
        <main class="map-container glass-panel">
            <div id="cityMap"></div>
            <div class="map-overlay">
                <div class="status-indicator">
                    <span class="dot"></span> System Active
                </div>
            </div>
        </main>
    </div>

    <!-- Leaflet JS -->
    <script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"
        integrity="sha256-20nQCchB9co0qIjJZRGuk2/Z9VM+kNiyxNV1lvTlZBo=" crossorigin=""></script>
    <script src="{{ url_for('static', filename='js/script.js') }}"></script>
</body>

</html>"""
with open(file_path, 'w', encoding='utf-8') as f:
    f.write(content)
print(f'Successfully generated {file_path}')

Successfully generated templates/dashboard.html


In [5]:
import os
os.makedirs('static/css', exist_ok=True)
file_path = os.path.join('static/css', 'style.css')
content = """:root {
    --bg-color: #0f172a;
    --primary-color: #10b981; /* Emerald Green */
    --secondary-color: #3b82f6; /* Blue */
    --accent-color: #f59e0b; /* Amber */
    --glass-bg: rgba(30, 41, 59, 0.7);
    --glass-border: rgba(255, 255, 255, 0.1);
    --text-color: #f8fafc;
    --text-muted: #94a3b8;
    --danger-color: #ef4444;
}

* {
    margin: 0;
    padding: 0;
    box-sizing: border-box;
    font-family: 'Outfit', sans-serif;
}

body {
    background-color: var(--bg-color);
    color: var(--text-color);
    height: 100vh;
    overflow: hidden;
}

/* Background Animation */
.background-globes {
    position: absolute;
    top: 0;
    left: 0;
    width: 100%;
    height: 100%;
    z-index: -1;
    overflow: hidden;
}

.globe {
    position: absolute;
    border-radius: 50%;
    filter: blur(80px);
    opacity: 0.6;
}

.globe-1 {
    width: 300px;
    height: 300px;
    background: var(--primary-color);
    top: -50px;
    left: -50px;
    animation: float 10s infinite ease-in-out;
}

.globe-2 {
    width: 400px;
    height: 400px;
    background: var(--secondary-color);
    bottom: -100px;
    right: -100px;
    animation: float 12s infinite ease-in-out reverse;
}

.globe-3 {
    width: 200px;
    height: 200px;
    background: var(--accent-color);
    top: 50%;
    left: 40%;
    animation: float 15s infinite ease-in-out;
}

@keyframes float {
    0%, 100% { transform: translate(0, 0); }
    50% { transform: translate(20px, -20px); }
}

/* Glass Panel Utility */
.glass-panel {
    background: var(--glass-bg);
    backdrop-filter: blur(12px);
    -webkit-backdrop-filter: blur(12px);
    border: 1px solid var(--glass-border);
    border-radius: 16px;
    box-shadow: 0 4px 30px rgba(0, 0, 0, 0.1);
}

/* Utility Classes */
.hidden { display: none !important; }

/* Login Page */
.login-body {
    display: flex;
    justify-content: center;
    align-items: center;
}

.login-container {
    width: 100%;
    max-width: 400px;
    padding: 2.5rem;
    text-align: center;
    transition: all 0.3s ease;
}

.login-header h1 {
    font-size: 2rem;
    margin-bottom: 0.5rem;
    background: linear-gradient(to right, var(--primary-color), var(--secondary-color));
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}

.login-header p {
    color: var(--text-muted);
    margin-bottom: 2rem;
    font-size: 0.9rem;
}

.form-toggle {
    background: rgba(0,0,0,0.2);
    border-radius: 8px;
    padding: 4px;
    display: flex;
    margin-bottom: 2rem;
}

.toggle-btn {
    flex: 1;
    background: none;
    border: none;
    color: var(--text-muted);
    padding: 8px;
    cursor: pointer;
    border-radius: 6px;
    transition: all 0.3s;
}

.toggle-btn.active {
    background: var(--glass-border);
    color: var(--text-color);
    font-weight: 600;
}

.auth-form {
    display: flex;
    flex-direction: column;
    gap: 1rem;
}

.input-group {
    position: relative;
}

.input-group i {
    position: absolute;
    left: 1rem;
    top: 50%;
    transform: translateY(-50%);
    color: var(--text-muted);
}

.input-group input {
    width: 100%;
    padding: 12px 12px 12px 40px;
    background: rgba(0,0,0,0.2);
    border: 1px solid var(--glass-border);
    border-radius: 8px;
    color: var(--text-color);
    outline: none;
    transition: border-color 0.3s;
}

.input-group input:focus {
    border-color: var(--primary-color);
}

.cta-button {
    background: linear-gradient(135deg, var(--primary-color), var(--secondary-color));
    color: white;
    border: none;
    padding: 12px;
    border-radius: 8px;
    font-weight: 600;
    cursor: pointer;
    transition: transform 0.2s;
}

.cta-button:hover {
    transform: translateY(-2px);
}

.cta-button.secondary {
    background: linear-gradient(135deg, var(--secondary-color), #6366f1);
}

.message {
    margin-top: 1rem;
    font-size: 0.9rem;
    min-height: 20px;
}

/* Dashboard */
.dashboard-body {
    display: flex;
    flex-direction: column;
}

.navbar {
    display: flex;
    justify-content: space-between;
    align-items: center;
    padding: 1rem 2rem;
    margin: 1rem;
    margin-bottom: 0;
}

.nav-brand {
    font-size: 1.25rem;
    font-weight: 700;
    color: var(--primary-color);
    display: flex;
    align-items: center;
    gap: 0.5rem;
}

.nav-user {
    display: flex;
    align-items: center;
    gap: 1rem;
}

.logout-btn {
    color: var(--text-muted);
    text-decoration: none;
    transition: color 0.3s;
}

.logout-btn:hover { color: var(--danger-color); }

.dashboard-container {
    display: grid;
    grid-template-columns: 350px 1fr;
    gap: 1rem;
    padding: 1rem;
    flex: 1;
    overflow: hidden;
}

.sidebar {
    padding: 1.5rem;
    display: flex;
    flex-direction: column;
    gap: 1.5rem;
    overflow-y: auto;
}

.control-group label {
    display: block;
    margin-bottom: 0.5rem;
    font-size: 0.9rem;
    color: var(--text-muted);
}

.input-wrapper {
    position: relative;
}

.input-wrapper i {
    position: absolute;
    left: 1rem;
    top: 50%;
    transform: translateY(-50%);
    color: var(--primary-color);
}

.input-wrapper input {
    width: 100%;
    padding: 12px 12px 12px 40px;
    background: rgba(0,0,0,0.2);
    border: 1px solid var(--glass-border);
    border-radius: 8px;
    color: var(--text-color);
    outline: none;
}

.activate-btn {
    background: linear-gradient(135deg, #ef4444, #f97316);
    color: white;
    border: none;
    padding: 1rem;
    border-radius: 8px;
    font-weight: 700;
    cursor: pointer;
    box-shadow: 0 4px 15px rgba(239, 68, 68, 0.3);
    transition: all 0.3s;
    display: flex;
    justify-content: center;
    align-items: center;
    gap: 0.5rem;
}

.activate-btn:hover {
    box-shadow: 0 6px 20px rgba(239, 68, 68, 0.5);
    transform: translateY(-2px);
}

.stats-panel {
    background: rgba(0,0,0,0.2);
    border-radius: 12px;
    padding: 1rem;
    border: 1px solid var(--glass-border);
}

.stats-panel h3 {
    font-size: 1rem;
    margin-bottom: 1rem;
    color: var(--secondary-color);
    border-bottom: 1px solid var(--glass-border);
    padding-bottom: 0.5rem;
}

.stat-item {
    display: flex;
    justify-content: space-between;
    margin-bottom: 0.5rem;
    font-size: 0.9rem;
}

.stat-item .highlight {
    color: var(--primary-color);
}

.logs-panel {
    flex: 1;
    background: rgba(0,0,0,0.3);
    border-radius: 8px;
    padding: 0.5rem;
    font-family: monospace;
    font-size: 0.8rem;
    color: var(--text-muted);
    overflow-y: auto;
    max-height: 200px;
}

.log-entry {
    margin-bottom: 4px;
    border-bottom: 1px solid rgba(255,255,255,0.05);
    padding-bottom: 4px;
}

.log-entry span.green { color: var(--primary-color); }
.log-entry span.red { color: var(--danger-color); }
.log-entry span.amber { color: var(--accent-color); }

/* Map */
.map-container {
    padding: 0;
    overflow: hidden;
    position: relative;
    border-radius: 16px;
}

#cityMap {
    width: 100%;
    height: 100%;
    background: #1e293b; /* Fallback */
}

/* Map Overlay */
.map-overlay {
    position: absolute;
    top: 1rem;
    right: 1rem;
    z-index: 1000;
}

.status-indicator {
    background: rgba(0,0,0,0.6);
    backdrop-filter: blur(4px);
    padding: 0.5rem 1rem;
    border-radius: 20px;
    display: flex;
    align-items: center;
    gap: 0.5rem;
    font-size: 0.8rem;
    border: 1px solid var(--primary-color);
    color: var(--primary-color);
}

.dot {
    width: 8px;
    height: 8px;
    background-color: var(--primary-color);
    border-radius: 50%;
    box-shadow: 0 0 10px var(--primary-color);
    animation: pulse 2s infinite;
}

@keyframes pulse {
    0% { opacity: 1; }
    50% { opacity: 0.5; }
    100% { opacity: 1; }
}

/* Leaflet Customization */
.leaflet-container {
    background: #0f172a !important;
    font-family: 'Outfit', sans-serif !important;
}

/* Scrollbar */
::-webkit-scrollbar {
    width: 6px;
}
::-webkit-scrollbar-track {
    background: transparent;
}
::-webkit-scrollbar-thumb {
    background: rgba(255,255,255,0.1);
    border-radius: 3px;
}
::-webkit-scrollbar-thumb:hover {
    background: rgba(255,255,255,0.2);
}
"""
with open(file_path, 'w', encoding='utf-8') as f:
    f.write(content)
print(f'Successfully generated {file_path}')

Successfully generated static/css/style.css


In [6]:
import os
os.makedirs('static/js', exist_ok=True)
file_path = os.path.join('static/js', 'script.js')
content = """document.addEventListener('DOMContentLoaded', () => {
    // --- Login Page Logic ---
    if (document.body.classList.contains('login-body')) {
        setupLogin();
    }

    // --- Dashboard Logic ---
    if (document.body.classList.contains('dashboard-body')) {
        setupDashboard();
    }
});

// --- LOGIN FUNCTIONS ---
function setupLogin() {
    const loginForm = document.getElementById('loginForm');
    if (loginForm) {
        loginForm.addEventListener('submit', async (e) => {
            e.preventDefault();
            const username = document.getElementById('loginUsername').value;
            const password = document.getElementById('loginPassword').value;

            showMessage('Authenticating...', 'amber');

            try {
                const res = await fetch('/login', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ username, password })
                });
                const data = await res.json();

                if (data.success) {
                    showMessage('Access Granted', 'green');
                    setTimeout(() => window.location.href = '/dashboard', 1000);
                } else {
                    showMessage(data.message || 'Login Failed', 'red');
                }
            } catch (err) {
                showMessage('Connection Error', 'red');
            }
        });
    }

    const registerForm = document.getElementById('registerForm');
    if (registerForm) {
        registerForm.addEventListener('submit', async (e) => {
            e.preventDefault();
            const username = document.getElementById('regUsername').value;
            const password = document.getElementById('regPassword').value;

            showMessage('Registering...', 'amber');

            try {
                const res = await fetch('/register', {
                    method: 'POST',
                    headers: { 'Content-Type': 'application/json' },
                    body: JSON.stringify({ username, password })
                });
                const data = await res.json();

                if (data.success) {
                    showMessage('Account Created. Please Login.', 'green');
                    showForm('login');
                } else {
                    showMessage(data.message || 'Registration Failed', 'red');
                }
            } catch (err) {
                showMessage('Connection Error', 'red');
            }
        });
    }
}

window.showForm = function (type) {
    const loginForm = document.getElementById('loginForm');
    const registerForm = document.getElementById('registerForm');
    const loginBtn = document.getElementById('loginBtn');
    const registerBtn = document.getElementById('registerBtn');

    if (type === 'login') {
        loginForm.classList.remove('hidden');
        registerForm.classList.add('hidden');
        loginBtn.classList.add('active');
        registerBtn.classList.remove('active');
    } else {
        loginForm.classList.add('hidden');
        registerForm.classList.remove('hidden');
        loginBtn.classList.remove('active');
        registerBtn.classList.add('active');
    }
    document.getElementById('message').textContent = '';
}

function showMessage(msg, type) {
    const el = document.getElementById('message');
    el.textContent = msg;
    el.style.color = type === 'red' ? '#ef4444' : (type === 'green' ? '#10b981' : '#f59e0b');
}

// --- DASHBOARD FUNCTIONS ---
let map;
let markers = {};
let routePolyline;

async function setupDashboard() {
    initMap();
    await loadLocations();
    await loadAccuracy();

    document.getElementById('activateSystemBtn').addEventListener('click', calculateRoute);
}

async function loadAccuracy() {
    try {
        const res = await fetch('/api/accuracy');
        const data = await res.json();
        document.getElementById('modelAcc').textContent = data.accuracy;
    } catch (e) {
        console.error("Failed to load accuracy");
    }
}

function initMap() {
    // Coordinate system is abstract (x, y), so we map it to Leaflet CRS.Simple
    map = L.map('cityMap', {
        crs: L.CRS.Simple,
        minZoom: -2,
        maxZoom: 2,
        zoomControl: true
    });

    // Roughly center on graph bounds (0,0 to 15,15) with some padding
    var bounds = [[-2, -2], [16, 16]];
    map.fitBounds(bounds);
}

const COORD_SCALE = 60; // Scale up coordinates to spread them out on the map

async function loadLocations() {
    try {
        const res = await fetch('/api/locations');
        const locations = await res.json();

        const datalist = document.getElementById('locationsList');
        const latLngs = [];

        locations.forEach(loc => {
            // Add to datalist
            const option = document.createElement('option');
            option.value = loc.name;
            datalist.appendChild(option);

            // Scale Coordinates for better visibility in CRS.Simple
            // Original: (x, y) roughly 0-15
            // Leaflet: [y, x]
            const y = loc.coords[1] * COORD_SCALE;
            const x = loc.coords[0] * COORD_SCALE;
            const latLng = [y, x];
            latLngs.push(latLng);

            // Add marker to map
            const marker = L.circleMarker(latLng, {
                radius: 12, // Larger marker
                fillColor: "#3b82f6",
                color: "#1e293b",
                weight: 2,
                opacity: 1,
                fillOpacity: 0.9
            }).addTo(map);

            marker.bindTooltip(loc.name, {
                permanent: true,
                direction: 'right', // Put text to the side
                className: 'map-tooltip',
                offset: [10, 0]
            });

            markers[loc.name] = marker;
        });

        // Auto-fit bounds to show all markers clearly
        if (latLngs.length > 0) {
            const bounds = L.latLngBounds(latLngs);
            map.fitBounds(bounds, { padding: [100, 100] });
        }

    } catch (err) {
        console.error("Failed to load locations", err);
    }
}

async function calculateRoute() {
    const start = document.getElementById('startLocation').value;
    const end = document.getElementById('endLocation').value;
    const logs = document.getElementById('routeLogs');
    const stats = document.getElementById('routeStats');

    if (!start || !end) {
        alert("Please enter both locations");
        return;
    }

    logs.innerHTML = '<div class="log-entry">🔄 Analyzing traffic patterns...</div>';

    try {
        const res = await fetch('/api/route', {
            method: 'POST',
            headers: { 'Content-Type': 'application/json' },
            body: JSON.stringify({ start, end })
        });
        const data = await res.json();

        if (!data.success) {
            logs.innerHTML += `<div class="log-entry"><span class="red">❌ ${data.message}</span></div>`;
            return;
        }

        // Success - Visuals
        document.getElementById('startLocation').value = data.start_corrected;
        document.getElementById('endLocation').value = data.end_corrected;

        logs.innerHTML += `<div class="log-entry">✅ Route Found: ${data.start_corrected} <i class="fa-solid fa-arrow-right"></i> ${data.end_corrected}</div>`;
        logs.innerHTML += `<div class="log-entry"><span class="amber">🚑 EMERGENCY VEHICLE DETECTED</span></div>`;

        // Update Stats
        document.getElementById('signalsCount').textContent = data.signals_cleared;
        document.getElementById('timeSaved').textContent = data.saved_time + ' mins';
        document.getElementById('newTime').textContent = data.new_time + ' mins';

        stats.classList.remove('hidden');

        // Draw Route
        if (routePolyline) map.removeLayer(routePolyline);

        const pathCoords = data.path.map(nodeName => {
            const m = markers[nodeName];
            return m.getLatLng();
        });

        routePolyline = L.polyline(pathCoords, {
            color: '#10b981', // Emerald
            weight: 6,
            dashArray: '15, 15',
            opacity: 0.9,
            lineJoin: 'round'
        }).addTo(map);

        // Animate Dash (Green Corridor effect)
        let offset = 0;
        if (window.routeAnim) clearInterval(window.routeAnim);
        window.routeAnim = setInterval(() => {
            offset = (offset - 2) % 30; // Faster animation
            routePolyline.setStyle({ dashOffset: offset });
        }, 50);

        map.fitBounds(routePolyline.getBounds(), { padding: [100, 100], maxZoom: 1 });

        // Detailed Logs (Simulate real-time)
        data.details.forEach((step, index) => {
            setTimeout(() => {
                let colorClass = step.traffic === 'HIGH' ? 'red' : (step.traffic === 'MEDIUM' ? 'amber' : 'green');
                let html = `<div class="log-entry">
                    🚦 Signal at ${step.node}: Traffic <span class="${colorClass}">${step.traffic}</span>
                    <i class="fa-solid fa-arrow-right"></i> <span class="green">CLEARED</span>
                    (Saved ${step.saved}m)
                </div>`;
                logs.innerHTML += html;
                logs.scrollTop = logs.scrollHeight;

                // Highlight Marker
                if (markers[step.node]) {
                    const originalRadius = markers[step.node].options.radius;
                    markers[step.node].setStyle({ fillColor: '#10b981', radius: originalRadius + 5, color: '#fff', weight: 4 });
                    setTimeout(() => markers[step.node].setStyle({ fillColor: '#3b82f6', radius: originalRadius, color: '#1e293b', weight: 2 }), 1500);
                }

            }, index * 800); // Stagger by 800ms
        });

    } catch (err) {
        logs.innerHTML += `<div class="log-entry"><span class="red">❌ System Error</span></div>`;
        console.error(err);
    }
}
"""
with open(file_path, 'w', encoding='utf-8') as f:
    f.write(content)
print(f'Successfully generated {file_path}')

Successfully generated static/js/script.js


In [7]:
# Run the Flask App
if __name__ == '__main__':
    # Using use_reloader=False is crucial in notebooks to prevent infinite loops
    app.run(debug=True, port=5000, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
